# 📦 Notebook 01: Solver Clásico del Problema de la Mochila

**Herramienta:** PuLP + Solver CBC (gratuito)  
**Objetivo:** Modelar y resolver el Knapsack Problem como un programa lineal entero (ILP)  
**Tiempo estimado:** 15 minutos

---

## 1. Formulación Matemática

El **Problema de la Mochila (Knapsack)** se formula como:

$$\max \sum_{i=1}^{n} v_i \, x_i$$

Sujeto a:

$$\sum_{i=1}^{n} w_i \, x_i \leq W$$

$$x_i \in \{0, 1\} \quad \forall \, i = 1, \ldots, n$$

Donde:
- $v_i$: valor del ítem $i$
- $w_i$: peso del ítem $i$
- $W$: capacidad máxima de la mochila
- $x_i$: variable binaria (1 = seleccionado, 0 = no seleccionado)

## 2. Importar Librerías

In [ ]:
import json
import pulp
import matplotlib.pyplot as plt
import numpy as np

print(f"PuLP versión: {pulp.VERSION}")
print(f"Solver disponible: CBC (incluido con PuLP)")

## 3. Cargar la Instancia del Problema

Usamos la misma instancia en los 3 notebooks para poder comparar resultados.

In [ ]:
# Cargar datos desde el archivo JSON compartido
with open('../data/knapsack_instance.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

# Extraer parámetros
items = data['items']
W = data['capacidad']       # Capacidad máxima
n = len(items)              # Número de ítems

valores = [item['valor'] for item in items]
pesos   = [item['peso']  for item in items]
nombres = [item['nombre'] for item in items]

# Mostrar la instancia
print(f"Problema de la Mochila - {n} ítems, capacidad W = {W}")
print(f"{'─'*50}")
for item in items:
    print(f"  Ítem {item['id']}: {item['nombre']:10s}  │  valor = {item['valor']:2d}  │  peso = {item['peso']}")
print(f"{'─'*50}")

## 4. Construir el Modelo con PuLP

### 4.1 Crear el problema de optimización

In [ ]:
# Crear el problema (maximización)
modelo = pulp.LpProblem("Mochila_Clasica", pulp.LpMaximize)

# Definir variables de decisión binarias: x_i ∈ {0, 1}
x = [pulp.LpVariable(f"x_{i}", cat='Binary') for i in range(n)]

print("Variables de decisión creadas:")
for i in range(n):
    print(f"  x_{i} ({nombres[i]}): binaria ∈ {{0, 1}}")

### 4.2 Función objetivo

$$\max \; Z = 8x_1 + 10x_2 + 15x_3 + 4x_4 + 12x_5$$

In [ ]:
# Función objetivo: maximizar el valor total
modelo += pulp.lpSum([valores[i] * x[i] for i in range(n)]), "Valor_Total"

print(f"Función objetivo: max Z = {' + '.join(f'{valores[i]}·x{i+1}' for i in range(n))}")

### 4.3 Restricción de capacidad

$$3x_1 + 5x_2 + 7x_3 + 2x_4 + 6x_5 \leq 15$$

In [ ]:
# Restricción: el peso total no debe exceder la capacidad
modelo += pulp.lpSum([pesos[i] * x[i] for i in range(n)]) <= W, "Restriccion_Capacidad"

print(f"Restricción: {' + '.join(f'{pesos[i]}·x{i+1}' for i in range(n))} ≤ {W}")
print(f"\nModelo completo:")
print(modelo)

## 5. Resolver el Modelo

In [ ]:
# Resolver con el solver CBC (incluido gratuitamente con PuLP)
modelo.solve(pulp.PULP_CBC_CMD(msg=0))

# Verificar estado
print(f"Estado de la solución: {pulp.LpStatus[modelo.status]}")
print(f"{'═'*50}")
print(f"\n🎯 VALOR ÓPTIMO: {int(pulp.value(modelo.objective))}")
print(f"\nÍtems seleccionados:")

peso_total = 0
valor_total = 0
for i in range(n):
    seleccionado = int(x[i].varValue)
    marca = '✅' if seleccionado else '  '
    print(f"  {marca} x_{i+1} = {seleccionado}  │  {nombres[i]:10s}  │  valor={valores[i]:2d}  │  peso={pesos[i]}")
    if seleccionado:
        peso_total += pesos[i]
        valor_total += valores[i]

print(f"\n{'─'*50}")
print(f"  Valor total: {valor_total}")
print(f"  Peso total:  {peso_total} / {W}")

## 6. Visualización de Resultados

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Gráfica 1: Selección de ítems ---
seleccion = [int(x[i].varValue) for i in range(n)]
colores = ['#3B82F6' if s else '#E2E8F0' for s in seleccion]
bordes  = ['#1E3A8A' if s else '#CBD5E1' for s in seleccion]

bars = axes[0].bar(nombres, valores, color=colores, edgecolor=bordes, linewidth=2)
axes[0].set_ylabel('Valor', fontsize=12)
axes[0].set_title('Selección Óptima de Ítems', fontsize=14, fontweight='bold')
axes[0].set_ylim(0, max(valores) * 1.2)

for i, (bar, sel) in enumerate(zip(bars, seleccion)):
    if sel:
        axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                     '✓', ha='center', fontsize=16, color='#059669')

# --- Gráfica 2: Uso de capacidad ---
axes[1].barh(['Usado', 'Disponible'], [peso_total, W - peso_total],
             color=['#3B82F6', '#E2E8F0'], edgecolor=['#1E3A8A', '#CBD5E1'], linewidth=2)
axes[1].set_xlabel('Peso', fontsize=12)
axes[1].set_title(f'Uso de Capacidad ({peso_total}/{W})', fontsize=14, fontweight='bold')
axes[1].set_xlim(0, W * 1.1)

plt.tight_layout()
plt.show()

## 7. Conclusiones

| Aspecto | Detalle |
|---------|--------|
| **Solver** | CBC (COIN-OR Branch and Cut) |
| **Tipo** | Exacto (garantiza solución óptima) |
| **Complejidad** | Exponencial en el peor caso (NP-hard) |
| **Escalabilidad** | Funciona bien hasta ~miles de variables |

### Siguiente paso
En el **Notebook 02**, resolveremos este mismo problema usando **QAOA con Qiskit 2.4** (computación cuántica basada en compuertas).